# 2. Data Processing and Transformation

This notebook loads the raw data cached by `1_data_download.ipynb` and applies the necessary transformations to prepare it for econometric analysis. The key steps are:

1.  **Setup**: Import necessary libraries.
2.  **Load Data**: Read the cached CSV files into pandas DataFrames.
3.  **Define Reusable Functions**: Create functions for common transformations (log-differencing, first-differencing) to ensure consistency and reduce code duplication.
4.  **Apply Transformations**: Process each variable in the specified order (`indpro`, `cpi`, `fedfunds`, `permits`, `unemp`, `hpi`), applying the appropriate transformation.
5.  **Store Transformed Data**: Collect all raw and transformed data into a structured dictionary for easy access in subsequent notebooks.
6.  **Verification**: Display the contents of the final dictionary to confirm success.

In [1]:
# Part 1: Setup
# =================
# This cell imports the necessary libraries for data manipulation.

import pandas as pd
import numpy as np
import os

In [2]:
# Part 2: Load Raw Data
# ========================
# This cell loads the raw time series data that was downloaded and cached in the
# previous notebook. It reads each CSV file from the 'data_cache' directory
# into a pandas DataFrame and stores them in a dictionary named 'data'.

data = {}
cache_dir = 'data_cache'
# Define the areas to load, which correspond to the filenames in the cache
areas_to_load = ['National', 'SF', 'DET', 'ATL', 'LV']

for area in areas_to_load:
    cache_path = os.path.join(cache_dir, f'{area}_data.csv')
    if os.path.exists(cache_path):
        print(f"Loading cached data for {area}...")
        # Load the CSV, setting the 'date' column as the index
        data[area] = pd.read_csv(cache_path, index_col='date', parse_dates=True)
    else:
        print(f"Cache file not found for {area} at {cache_path}. Please run the data download notebook first.")

print("\\nData loading complete.")

# --- Verification ---
# Display the first few rows of one of the DataFrames to confirm successful loading
if 'SF' in data:
    print("\\nSample of raw data for SF:")
    print(data['SF'].head())

Loading cached data for National...
Loading cached data for SF...
Loading cached data for DET...
Loading cached data for ATL...
Loading cached data for LV...
\nData loading complete.
\nSample of raw data for SF:
             indpro    cpi  fedfunds        hpi  unemp      permits
date                                                               
1991-01-01  61.0842  134.7      6.91  71.935495    4.6  2037.040992
1991-02-01  60.6387  134.8      6.25  71.126390    4.7   585.889721
1991-03-01  60.2973  134.8      6.12  70.608208    4.9   835.829789
1991-04-01  60.4524  135.1      5.91  69.909268    5.0   789.687733
1991-05-01  61.0587  135.6      5.78  70.010559    5.1   625.659862


In [3]:
# Part 3: Reusable Transformation Functions
# =========================================
# To ensure consistency and reduce repetition, we define reusable functions
# for our primary transformations: log-differencing and first-differencing.

def log_difference(series: pd.Series, drop_na: bool = True) -> pd.Series:
    """
    Calculates the log-difference of a pandas Series.
    This is often used to approximate the percentage change of a time series.
    
    Args:
        series (pd.Series): The input time series.
        drop_na (bool): Whether to drop NaN values resulting from the difference.
        
    Returns:
        pd.Series: The log-differenced series.
    """
    transformed = np.log(series).diff()
    return transformed.dropna() if drop_na else transformed

def first_difference(series: pd.Series, drop_na: bool = True) -> pd.Series:
    """
    Calculates the first-difference of a pandas Series.
    This represents the change from one period to the next.
    
    Args:
        series (pd.Series): The input time series.
        drop_na (bool): Whether to drop NaN values resulting from the difference.
        
    Returns:
        pd.Series: The first-differenced series.
    """
    transformed = series.diff()
    return transformed.dropna() if drop_na else transformed

In [4]:
# Part 4: Apply Transformations
# ==============================
# This section applies the defined transformations to the raw data.
# The variables are processed in the specified Cholesky decomposition order.
# All resulting series are stored in a new dictionary for easy access.

transformed_data = {}

# Store raw dataframes for reference
for area, df in data.items():
    transformed_data[f'{area}_raw'] = df

### 4.1 Industrial Production (indpro)

We apply a log-difference transformation to the Industrial Production series to stabilize its variance and mean.

In [5]:
if 'National_raw' in transformed_data and 'indpro' in transformed_data['National_raw']:
    # Store the raw series
    transformed_data['indpro_raw'] = transformed_data['National_raw']['indpro']
    # Transform and store the log-differenced series
    transformed_data['indpro_log_diff'] = log_difference(transformed_data['National_raw']['indpro'])
    print("Processed 'indpro' (raw and log-difference).")
else:
    print("'indpro' data not found.")

Processed 'indpro' (raw and log-difference).


### 4.2 Consumer Price Index (cpi)

We apply a log-difference transformation to the Consumer Price Index, which serves as a proxy for the inflation rate.

In [6]:
if 'National_raw' in transformed_data and 'cpi' in transformed_data['National_raw']:
    # Store the raw series
    transformed_data['cpi_raw'] = transformed_data['National_raw']['cpi']
    # Transform and store the log-differenced series
    transformed_data['cpi_log_diff'] = log_difference(transformed_data['National_raw']['cpi'])
    print("Processed 'cpi' (raw and log-difference).")
else:
    print("'cpi' data not found.")

Processed 'cpi' (raw and log-difference).


### 4.3 Federal Funds Rate (fedfunds)

The Federal Funds Rate is made stationary by taking its first-difference.

In [7]:
if 'National_raw' in transformed_data and 'fedfunds' in transformed_data['National_raw']:
    # Store the raw series
    transformed_data['fedfunds_raw'] = transformed_data['National_raw']['fedfunds']
    # Transform and store the first-differenced series
    transformed_data['fedfunds_diff'] = first_difference(transformed_data['National_raw']['fedfunds'])
    print("Processed 'fedfunds' (raw and first-difference).")
else:
    print("'fedfunds' data not found.")

Processed 'fedfunds' (raw and first-difference).


### 4.4 Building Permits (permits)

Building permits for each local area are transformed using log-differencing.

In [8]:
# Store the raw series
permits_raw_df = pd.DataFrame({area: transformed_data[f'{area}_raw']['permits'] for area in areas_to_load if f'{area}_raw' in transformed_data and 'permits' in transformed_data[f'{area}_raw']})
if not permits_raw_df.empty:
    transformed_data['permits_raw'] = permits_raw_df

# Transform and store the log-differenced series
permits_log_diff_df = pd.DataFrame()
for area in areas_to_load:
    if f'{area}_raw' in transformed_data and 'permits' in transformed_data[f'{area}_raw']:
        permits_log_diff_df[area] = log_difference(transformed_data[f'{area}_raw']['permits'])

if not permits_log_diff_df.empty:
    transformed_data['permits_log_diff'] = permits_log_diff_df.dropna(how='all')
    print("Processed 'permits' for all areas (raw and log-difference).")
else:
    print("'permits' data not found for any area.")

Processed 'permits' for all areas (raw and log-difference).


### 4.5 Unemployment (unemp)

The unemployment rate for each local area is made stationary by taking its first-difference.

In [9]:
# Store the raw series
unemp_raw_df = pd.DataFrame({area: transformed_data[f'{area}_raw']['unemp'] for area in areas_to_load if f'{area}_raw' in transformed_data and 'unemp' in transformed_data[f'{area}_raw']})
if not unemp_raw_df.empty:
    transformed_data['unemp_raw'] = unemp_raw_df

# Transform and store the first-differenced series
unemp_diff_df = pd.DataFrame()
for area in areas_to_load:
    if f'{area}_raw' in transformed_data and 'unemp' in transformed_data[f'{area}_raw']:
        unemp_diff_df[area] = first_difference(transformed_data[f'{area}_raw']['unemp'])

if not unemp_diff_df.empty:
    transformed_data['unemp_diff'] = unemp_diff_df.dropna(how='all')
    print("Processed 'unemp' for all areas (raw and first-difference).")
else:
    print("'unemp' data not found for any area.")

Processed 'unemp' for all areas (raw and first-difference).


### 4.6 House Price Index (hpi)

The House Price Index for each local area is transformed using log-differencing to approximate the housing price growth rate.

In [10]:
# Store the raw series
hpi_raw_df = pd.DataFrame({area: transformed_data[f'{area}_raw']['hpi'] for area in areas_to_load if f'{area}_raw' in transformed_data and 'hpi' in transformed_data[f'{area}_raw']})
if not hpi_raw_df.empty:
    transformed_data['hpi_raw'] = hpi_raw_df

# Transform and store the log-differenced series
hpi_log_diff_df = pd.DataFrame()
for area in areas_to_load:
    if f'{area}_raw' in transformed_data and 'hpi' in transformed_data[f'{area}_raw']:
        hpi_log_diff_df[area] = log_difference(transformed_data[f'{area}_raw']['hpi'])

if not hpi_log_diff_df.empty:
    transformed_data['hpi_log_diff'] = hpi_log_diff_df.dropna(how='all')
    print("Processed 'hpi' for all areas (raw and log-difference).")
else:
    print("'hpi' data not found for any area.")

Processed 'hpi' for all areas (raw and log-difference).


In [11]:
# Part 5: Verification
# ======================
# This cell confirms that all transformations have been completed and stored.

print("\\n--- Data Processing Complete ---")
print("The 'transformed_data' dictionary contains the following keys:")
for key in sorted(transformed_data.keys()):
    print(f"- {key}")

# Display a sample of a transformed DataFrame to verify the contents
if 'hpi_log_diff' in transformed_data:
    print("\\nSample of log-differenced HPI data:")
    print(transformed_data['hpi_log_diff'].head())

\n--- Data Processing Complete ---
The 'transformed_data' dictionary contains the following keys:
- ATL_raw
- DET_raw
- LV_raw
- National_raw
- SF_raw
- cpi_log_diff
- cpi_raw
- fedfunds_diff
- fedfunds_raw
- hpi_log_diff
- hpi_raw
- indpro_log_diff
- indpro_raw
- permits_log_diff
- permits_raw
- unemp_diff
- unemp_raw
\nSample of log-differenced HPI data:
            National        SF       DET       ATL        LV
date                                                        
1991-02-01 -0.003767 -0.011311 -0.004954 -0.002154  0.011794
1991-03-01 -0.001503 -0.007312  0.000376 -0.002089  0.001194
1991-04-01  0.002433 -0.009948  0.001050  0.005667  0.001028
1991-05-01  0.005944  0.001448  0.005114  0.001123  0.001340
1991-06-01  0.006433  0.003860  0.004976  0.003663 -0.001016
